# Process Midi
File to process midi files to remove corrupt files and extract features we're interested in. 

In [1]:
# Import packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

import pretty_midi
import librosa
import mir_eval
import mir_eval.display
import tables
import IPython.display
import os
import json

In [2]:
# Loop through files in raw_data folder to extract data from midi files and store in dataframe
folder = '../data/raw_data/lmd_full'

records = []
errors = []

for root, dirs, files in os.walk(folder):
    for file in files:
        if not file.endswith('.mid'):
            continue
        
        filepath = os.path.join(root, file)
        
        try:
            midi = pretty_midi.PrettyMIDI(filepath)
            
            tempo_times, tempos = midi.get_tempo_changes()
            key_changes = midi.key_signature_changes
            time_sig_changes = midi.time_signature_changes

            records.append({
                'file':               file,
                'filepath':           filepath,
                'duration_s':         round(midi.get_end_time(), 2),
                'resolution':         midi.resolution,
                'n_instruments':      len(midi.instruments),
                'n_notes':            sum(len(i.notes) for i in midi.instruments),
                'n_tempo_changes':    len(tempos),
                'initial_bpm':        round(tempos[0], 2) if len(tempos) > 0 else None,
                'n_key_changes':      len(key_changes),
                'initial_key':        key_changes[0].key_number if len(key_changes) > 0 else None,
                'n_time_sig_changes': len(time_sig_changes),
                'initial_time_sig':   f"{time_sig_changes[0].numerator}/{time_sig_changes[0].denominator}" if len(time_sig_changes) > 0 else None,
            })

        except Exception as e:
            errors.append({'file': filepath, 'error': str(e)})

    # Progress update per subfolder
    if records:
        print(f"Processed {len(records)} files so far (latest folder: {root})...")

df = pd.DataFrame(records)
print(f"\nDone. Successfully parsed: {len(df)} files")
print(f"Errors: {len(errors)} files")

/home/arige/miniconda3/envs/dsan2/lib/python3.11/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Processed 10966 files so far (latest folder: ../data/raw_data/lmd_full/d)...
Processed 21746 files so far (latest folder: ../data/raw_data/lmd_full/3)...
Processed 32711 files so far (latest folder: ../data/raw_data/lmd_full/f)...
Processed 43522 files so far (latest folder: ../data/raw_data/lmd_full/e)...
Processed 54513 files so far (latest folder: ../data/raw_data/lmd_full/7)...
Processed 65355 files so far (latest folder: ../data/raw_data/lmd_full/a)...
Processed 76204 files so far (latest folder: ../data/raw_data/lmd_full/8)...
Processed 87067 files so far (latest folder: ../data/raw_data/lmd_full/2)...
Processed 98027 files so far (latest folder: ../data/raw_data/lmd_full/b)...
Processed 108907 files so far (latest folder: ../data/raw_data/lmd_full/0)...
Processed 119780 files so far (latest folder: ../data/raw_data/lmd_full/4)...
Processed 130704 files so far (latest folder: ../data/raw_data/lmd_full/c)...
Processed 141553 files so far (latest folder: ../data/raw_data/lmd_full/9

In [ ]:
# Save
df.to_csv('../data/processed_data/lmd_full_metadata.csv', index=False) 
# Note: This saves the time signature as dates. When reading the csv, make sure to include parse_dates=False

In [ ]:
# Explore the remaining data
# Basic counts
print(f"Total files:       {len(df)}")
print(f"Successful parses: {df['initial_key'].notna().sum()}")
print(f"Missing key:       {df['initial_key'].isna().sum()}")
print(f"Missing time sig:  {df['initial_time_sig'].isna().sum()}")

# --- Duration ---
print("\n--- Duration (seconds) ---")
print(df['duration_s'].describe())

# --- BPM ---
print("\n--- BPM ---")
print(df['initial_bpm'].describe())

# --- Instruments per file ---
print("\n--- Instruments per file ---")
print(df['n_instruments'].describe())

# --- Notes per file ---
print("\n--- Notes per file ---")
print(df['n_notes'].describe())

# --- Time signature distribution ---
print("\n--- Time Signature Counts ---")
print(df['initial_time_sig'].value_counts().to_string())

# --- Key distribution ---
key_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def decode_key(key_number):
    if pd.isna(key_number):
        return None
    key_number = int(key_number)
    name = key_names[key_number % 12]
    mode = 'major' if key_number < 12 else 'minor'
    return f"{name} {mode}"

df['key_name'] = df['initial_key'].apply(decode_key)
print("\n--- Key Distribution ---")
print(df['key_name'].value_counts())

# --- Suspicious files ---
suspicious = df[
    (df['duration_s'] < 5) |
    (df['n_notes'] < 10) |
    (df['initial_bpm'] < 20) |
    (df['initial_bpm'] > 300) |
    (df['initial_key'].isna()) |
    (df['initial_time_sig'].isna())
]
print(f"\nSuspicious files: {len(suspicious)} ({100 * len(suspicious) / len(df):.1f}%)")

# --- Files with multiple tempo changes ---
complex_tempo = df[df['n_tempo_changes'] > 1]
print(f"Files with tempo changes: {len(complex_tempo)} ({100 * len(complex_tempo) / len(df):.1f}%)")

# --- Extremes worth spot-checking ---
print("\n--- Slowest 5 ---")
print(df.sort_values('initial_bpm').head()[['file', 'initial_bpm', 'key_name', 'initial_time_sig']])

print("\n--- Fastest 5 ---")
print(df.sort_values('initial_bpm').tail()[['file', 'initial_bpm', 'key_name', 'initial_time_sig']])

print("\n--- Shortest 5 ---")
print(df.sort_values('duration_s').head()[['file', 'duration_s', 'initial_bpm', 'key_name']])

print("\n--- Most instruments ---")
print(df.sort_values('n_instruments').tail()[['file', 'n_instruments', 'n_notes', 'key_name']])

Total files:       174421
Successful parses: 98165
Missing key:       76256
Missing time sig:  16671

--- Duration (seconds) ---
count    174421.000000
mean        193.647649
std         215.584243
min           0.000000
25%         119.990000
50%         191.560000
75%         246.880000
max       31055.620000
Name: duration_s, dtype: float64

--- BPM ---
count    174421.000000
mean        116.833576
std          36.595530
min           8.000000
25%          94.000000
50%         120.000000
75%         131.000000
max         960.000000
Name: initial_bpm, dtype: float64

--- Instruments per file ---
count    174421.000000
mean          8.412903
std           6.111786
min           0.000000
25%           4.000000
50%           8.000000
75%          11.000000
max         990.000000
Name: n_instruments, dtype: float64

--- Notes per file ---
count    174421.000000
mean       3893.960658
std        3369.898906
min           0.000000
25%        1279.000000
50%        3320.000000
75%        